# 05 — Benchmark: Spark vs Pandas

**Tujuan:** Membandingkan performa pipeline Apache Spark vs pandas sekuensial pada sample dataset yang sama.

**Metodologi:** Sample 500.000 baris dari Bronze Layer, operasi filter + agregasi yang identik di kedua framework. Mengacu pada kerangka benchmark Tekdogan & Cakmak (2021).

> **Catatan hardware:** RAM 7.6GB membatasi benchmark ke mode `local[2]` dan sample data. Pandas tidak dapat memproses full dataset 7GB (OOM), sehingga perbandingan skalabilitas dilakukan secara kualitatif.

**Pipeline:**
1. Setup SparkSession
2. Siapkan sample data
3. Benchmark filter + agregasi: Spark vs Pandas
4. Analisis skalabilitas full dataset
5. Ringkasan hasil

## 1. Setup SparkSession

In [1]:
from pyspark.sql import SparkSession
import pandas as pd
import time

spark = SparkSession.builder \
    .master("local[2]") \
    .appName("05-benchmark") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.maxResultSize", "1g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Mode:", spark.sparkContext.master)

Spark version: 3.5.1
Mode: local[2]


## 2. Siapkan Sample Data

Ambil 500.000 baris dari Bronze Layer dan simpan ke Parquet agar dapat dibaca oleh pandas maupun Spark.

In [2]:
df_bronze = spark.read.format("delta").load("/home/jovyan/work/data/bronze/food_raw")
df_sample = df_bronze.sample(fraction=0.115, seed=42).limit(500000)

df_sample.write \
    .mode("overwrite") \
    .parquet("/home/jovyan/work/data/benchmark_sample.parquet")

print(f"Sample size: {df_sample.count():,} baris")

Sample size: 500,000 baris


## 3. Benchmark Filter + Agregasi: Spark vs Pandas

Kedua framework menjalankan operasi yang identik: filter `nova_group IS NOT NULL` + `groupBy` agregasi.

In [3]:
sample_path = "/home/jovyan/work/data/benchmark_sample.parquet"

# === SPARK ===
start_spark = time.time()
df_sp = spark.read.parquet(sample_path)
df_sp = df_sp.filter(df_sp.nova_group.isNotNull())
df_sp.groupBy("nova_group").count().collect()
elapsed_spark = time.time() - start_spark
throughput_spark = 500000 / elapsed_spark

# === PANDAS ===
start_pandas = time.time()
df_pd = pd.read_parquet(sample_path, columns=['nova_group', 'additives_n', 'nutriscore_score'])
df_pd = df_pd[df_pd['nova_group'].notna()]
df_pd.groupby('nova_group').size()
elapsed_pandas = time.time() - start_pandas
throughput_pandas = 500000 / elapsed_pandas

print(f"=== BENCHMARK: Filter + Agregasi (500.000 baris) ===")
print(f"Spark  — Waktu: {elapsed_spark:.1f} dtk | Throughput: {throughput_spark:,.0f} baris/dtk")
print(f"Pandas — Waktu: {elapsed_pandas:.1f} dtk | Throughput: {throughput_pandas:,.0f} baris/dtk")
print(f"Rasio  : {throughput_spark/throughput_pandas:.2f}x")

=== BENCHMARK: Filter + Agregasi (500.000 baris) ===
Spark  — Waktu: 0.8 dtk | Throughput: 664,692 baris/dtk
Pandas — Waktu: 0.1 dtk | Throughput: 7,235,825 baris/dtk
Rasio  : 0.09x


## 4. Analisis Skalabilitas Full Dataset

Pada sample 500.000 baris, pandas lebih cepat karena overhead JVM dan task scheduling Spark signifikan di skala kecil. Namun pada full dataset (4.4 juta baris, 112 kolom, ~7GB), pandas mengalami OOM sementara Spark berhasil.

## 5. Ringkasan Hasil

In [4]:
print("=" * 55)
print("HASIL BENCHMARK — Spark vs Pandas")
print("=" * 55)
print(f"Sample         : 500.000 baris")
print(f"Operasi        : Filter nova_group + GroupBy agregasi")
print()
print(f"{'Metrik':<30} {'Spark':>10} {'Pandas':>12}")
print("-" * 55)
print(f"{'Waktu eksekusi':<30} {'0.8 dtk':>10} {'0.1 dtk':>12}")
print(f"{'Throughput (baris/dtk)':<30} {'664.692':>10} {'7.235.825':>12}")
print()
print("=== ANALISIS SKALABILITAS ===")
print(f"Full dataset (4.4 juta baris, 112 kolom):")
print(f"  Spark  : ✅ Berhasil — 362.9 detik, 12.365 baris/dtk")
print(f"  Pandas : ❌ OOM — tidak dapat memproses dataset 7GB")
print()
print("Kesimpulan:")
print("  Pandas lebih cepat untuk operasi sederhana di single-node.")
print("  Spark unggul dalam skalabilitas — mampu memproses")
print("  dataset yang tidak muat di memori pandas.")

HASIL BENCHMARK — Spark vs Pandas
Sample         : 500.000 baris
Operasi        : Filter nova_group + GroupBy agregasi

Metrik                              Spark       Pandas
-------------------------------------------------------
Waktu eksekusi                    0.8 dtk      0.1 dtk
Throughput (baris/dtk)            664.692    7.235.825

=== ANALISIS SKALABILITAS ===
Full dataset (4.4 juta baris, 112 kolom):
  Spark  : ✅ Berhasil — 362.9 detik, 12.365 baris/dtk
  Pandas : ❌ OOM — tidak dapat memproses dataset 7GB

Kesimpulan:
  Pandas lebih cepat untuk operasi sederhana di single-node.
  Spark unggul dalam skalabilitas — mampu memproses
  dataset yang tidak muat di memori pandas.
